# Notebook 15 — Multi-Agent System Eval

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/evals/15_multi_agent_system_eval.ipynb)

Single-agent metrics miss a distinct class of failures: weak handoffs, unnecessary delegation, broken coordination, and corrupted shared state. In this notebook we build a small multi-agent workflow benchmark and score those failure modes directly.

## What you will build

- a small planner → researcher → writer → reviewer workflow
- several workflow variants with realistic coordination mistakes
- first-principles metrics for handoff quality, delegation quality, and shared-state correctness
- coordination failure detectors for loops, skipped prerequisites, and stale reads
- trade-off tables that compare quality against latency, cost, and workflow complexity

Everything stays local, open-source, and notebook-native.

## Why multi-agent eval needs its own metrics

A multi-agent system can fail even when each individual agent looks competent in isolation.

- the planner can delegate to the wrong specialist
- one agent can hand off a vague or incomplete task
- two agents can duplicate work
- a writer can read stale state after the researcher updated the brief
- a reviewer can create an unnecessary loop that hurts latency without improving quality

That means we need to evaluate the **workflow**, not only the final answer.

In [ ]:
# --- Google Colab Setup ---
!pip install -q "transformers>=4.51.0" accelerate bitsandbytes torch

import csv
import json
import math
import random
import statistics
import time
from pathlib import Path
from typing import Any, Dict, List

import torch
from google.colab import drive
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

drive.mount('/content/drive')
CACHE_DIR = "/content/drive/MyDrive/models"
MODEL_NAME = "Qwen/Qwen3-8B"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, cache_dir=CACHE_DIR)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    quantization_config=quantization_config,
    cache_dir=CACHE_DIR,
    torch_dtype="auto",
)

def generate(messages, max_new_tokens=512, temperature=0.7, do_sample=True, top_k=20):
    if isinstance(messages, str):
        messages = [{"role": "user", "content": messages}]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )
    inputs = tokenizer([prompt], return_tensors="pt").to(model.device)
    output_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=temperature if do_sample else None,
        do_sample=do_sample,
        top_k=top_k,
        pad_token_id=tokenizer.pad_token_id,
    )
    generated_ids = output_ids[0][inputs.input_ids.shape[1]:]
    return tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

print(f"✓ Model loaded: {MODEL_NAME}")
print(f"  GPU memory used: {torch.cuda.memory_allocated() / 1024**3:.1f} GB")

## Step 1: Add notebook helpers

The model from the setup cell is optional for a later qualitative audit. The core evaluation harness below is plain Python.

In [ ]:
from collections import Counter, defaultdict
import json

random.seed(15)

ARTIFACT_DIR = Path("artifacts") / "notebook_15_multi_agent"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

def to_markdown_table(rows, columns):
    header = "| " + " | ".join(columns) + " |"
    divider = "| " + " | ".join(["---"] * len(columns)) + " |"
    body = []
    for row in rows:
        body.append("| " + " | ".join(str(row.get(column, "")) for column in columns) + " |")
    return "\n".join([header, divider, *body])

def normalize_text(text):
    cleaned = "".join(char.lower() if char.isalnum() else " " for char in str(text))
    return " ".join(cleaned.split())

def keyword_recall(text, keywords):
    if not keywords:
        return 1.0
    normalized = normalize_text(text)
    hits = sum(1 for keyword in keywords if normalize_text(keyword) in normalized)
    return round(hits / len(keywords), 3)

def mean_or_zero(values):
    values = list(values)
    return round(statistics.fmean(values), 3) if values else 0.0

def deep_copy(value):
    return json.loads(json.dumps(value))

print("Artifact directory:", ARTIFACT_DIR.resolve())

## Step 2: Define a tiny multi-agent benchmark

Each case specifies:

- the user request
- which roles are actually necessary
- the facts that must survive handoffs
- the expected answer keywords
- a few forbidden mistakes

One case is intentionally simple so we can catch **over-delegation**.

In [ ]:
workflow_cases = [
    {
        "id": "pto_note",
        "user_request": "Write a short note to interns about the PTO policy.",
        "audience": "interns",
        "research_topic": "pto",
        "provided_context": [],
        "required_roles": ["planner", "researcher", "writer", "reviewer"],
        "must_include": ["interns", "15 PTO days", "do not accrue PTO"],
        "expected_keywords": ["interns", "15 pto days", "do not accrue pto"],
        "forbidden_keywords": ["interns receive 15 pto days"],
        "response_prefix": "Hello interns,",
        "stale_fact": "Interns receive 15 PTO days.",
    },
    {
        "id": "incident_update",
        "user_request": "Prepare a short executive update about the database failover incident.",
        "audience": "executives",
        "research_topic": "incident_0421",
        "provided_context": [],
        "required_roles": ["planner", "researcher", "writer", "reviewer"],
        "must_include": ["database failover", "rollback", "12 minutes", "no customer data loss"],
        "expected_keywords": ["database failover", "rollback", "12 minutes", "no customer data loss"],
        "forbidden_keywords": ["45 minutes", "customer data loss"],
        "response_prefix": "Executive update:",
        "stale_fact": "The outage lasted 45 minutes and caused customer data loss.",
    },
    {
        "id": "travel_policy",
        "user_request": "Summarize the travel reimbursement policy for new hires.",
        "audience": "new hires",
        "research_topic": "travel_reimbursement",
        "provided_context": [],
        "required_roles": ["planner", "researcher", "writer", "reviewer"],
        "must_include": ["submit within 30 days", "manager approval", "receipts over $25"],
        "expected_keywords": ["30 days", "manager approval", "receipts over 25"],
        "forbidden_keywords": ["120 days", "no approval needed"],
        "response_prefix": "Travel policy summary:",
        "stale_fact": "Expenses can be submitted within 120 days and never need approval.",
    },
    {
        "id": "flash_update",
        "user_request": "Turn these confirmed facts into a two-sentence flash update for executives.",
        "audience": "executives",
        "research_topic": None,
        "provided_context": [
            "Rollback restored traffic in 12 minutes.",
            "No customer data loss occurred.",
            "Root cause was a bad migration.",
        ],
        "required_roles": ["planner", "writer", "reviewer"],
        "must_include": ["12 minutes", "no customer data loss", "bad migration"],
        "expected_keywords": ["12 minutes", "no customer data loss", "bad migration"],
        "forbidden_keywords": ["customer data loss"],
        "response_prefix": "Executive flash update:",
        "stale_fact": "Customer data loss occurred during the incident.",
    },
]

print("Workflow cases:", len(workflow_cases))
print("Cases requiring research:", sum(1 for case in workflow_cases if case["research_topic"]))

## Step 3: Declare role capabilities, handoff requirements, and source data

We keep the runtime explicit:

- role capabilities let us score delegation quality
- handoff requirements let us score whether a message is complete for the receiver
- small local knowledge stores let the researcher fetch deterministic facts

In [ ]:
ROLE_CAPABILITIES = {
    "planner": ["plan", "route", "delegate"],
    "researcher": ["research", "facts", "sources", "brief"],
    "writer": ["draft", "rewrite", "compose"],
    "reviewer": ["review", "check", "approve"],
}

HANDOFF_REQUIREMENTS = {
    "researcher": ["objective", "topic_or_context", "must_include", "success_criteria"],
    "writer": ["objective", "audience", "must_include", "source_keys"],
    "reviewer": ["objective", "rubric", "draft_key", "blocking_checks"],
}

ROLE_RUNTIME = {
    "planner": {"latency_s": 0.9, "cost_units": 0.010},
    "researcher": {"latency_s": 1.4, "cost_units": 0.015},
    "writer": {"latency_s": 1.6, "cost_units": 0.020},
    "reviewer": {"latency_s": 1.2, "cost_units": 0.015},
}

KNOWLEDGE_BASE = {
    "pto": {
        "facts": [
            "Full-time employees receive 15 PTO days.",
            "Interns do not accrue PTO.",
            "Questions go to HR, not payroll.",
        ],
        "source_docs": ["policy_handbook_v3"],
    },
    "incident_0421": {
        "facts": [
            "A database failover caused the incident.",
            "The team rolled back the bad change.",
            "Traffic normalized within 12 minutes.",
            "There was no customer data loss.",
        ],
        "source_docs": ["incident_0421_report"],
    },
    "travel_reimbursement": {
        "facts": [
            "Travel expenses must be submitted within 30 days.",
            "Manager approval is required for trips.",
            "Receipts are required for expenses over $25.",
        ],
        "source_docs": ["travel_policy_2026"],
    },
}

def objective_matches_role(role_name, objective):
    text = normalize_text(objective)
    return any(token in text for token in ROLE_CAPABILITIES[role_name])

def serialize_value(value):
    if isinstance(value, list):
        return ", ".join(str(item) for item in value)
    if isinstance(value, dict):
        return json.dumps(value, sort_keys=True)
    return str(value)

print("Roles:", ", ".join(ROLE_CAPABILITIES))
print("Knowledge topics:", ", ".join(KNOWLEDGE_BASE))

## Step 4: Build a deterministic workflow runner

We simulate five workflow styles:

- **disciplined**: clear handoffs and correct state use
- **vague_handoff**: the handoffs omit key details
- **over_delegating**: extra routing adds cost without adding value
- **state_desync**: the writer reads stale or corrupted shared state
- **reviewer_loop**: the reviewer creates a redundant revision cycle

In [ ]:
def make_note_text(fields):
    ordered_keys = sorted(fields)
    parts = []
    for key in ordered_keys:
        parts.append(f"{key}: {serialize_value(fields[key])}")
    return "; ".join(parts)

def build_handoff(from_role, to_role, fields):
    return {
        "from": from_role,
        "to": to_role,
        "fields": deep_copy(fields),
        "text": make_note_text(fields),
    }

def append_step(run, role, reads=None, writes=None, handoff=None, summary=""):
    reads = deep_copy(reads or {})
    writes = deep_copy(writes or {})
    state_before = deep_copy(run["state"])

    for key, value in writes.items():
        run["state"][key] = deep_copy(value)

    runtime = ROLE_RUNTIME[role]
    record = {
        "role": role,
        "reads": reads,
        "writes": writes,
        "handoff": deep_copy(handoff) if handoff else None,
        "summary": summary,
        "latency_s": runtime["latency_s"],
        "cost_units": runtime["cost_units"],
        "state_before": state_before,
        "state_after": deep_copy(run["state"]),
    }
    run["steps"].append(record)
    run["latency_s"] += runtime["latency_s"]
    run["cost_units"] += runtime["cost_units"]
    return record

def planner_handoff(case, variant, next_role, source_keys):
    if next_role == "researcher":
        fields = {
            "objective": "research the request and return a fact brief",
            "topic_or_context": case["research_topic"],
            "must_include": case["must_include"],
            "success_criteria": "return verified facts and source keys",
        }
    else:
        fields = {
            "objective": "draft the response",
            "audience": case["audience"],
            "must_include": case["must_include"],
            "source_keys": source_keys,
        }

    if variant == "vague_handoff":
        fields = {"objective": "help with this task"}

    return build_handoff("planner", next_role, fields)

def researcher_handoff(case, variant, source_keys):
    fields = {
        "objective": "draft the response",
        "audience": case["audience"],
        "must_include": case["must_include"],
        "source_keys": source_keys,
    }
    if variant == "vague_handoff":
        fields = {
            "objective": "write something concise",
            "source_keys": source_keys,
        }
    return build_handoff("researcher", "writer", fields)

def writer_handoff(case, variant):
    fields = {
        "objective": "review factual coverage and approve or block",
        "rubric": ["fact coverage", "audience fit", "no contradictions"],
        "draft_key": "draft",
        "blocking_checks": case["forbidden_keywords"],
    }
    if variant == "vague_handoff":
        fields = {"objective": "take a look"}
    return build_handoff("writer", "reviewer", fields)

def reviewer_revision_handoff(case):
    fields = {
        "objective": "rewrite the draft even though no blocking issue exists",
        "audience": case["audience"],
        "must_include": case["must_include"],
        "source_keys": ["draft", "review_notes"],
    }
    return build_handoff("reviewer", "writer", fields)

def build_draft(case, facts):
    return case["response_prefix"] + " " + " ".join(facts)

def review_draft(case, draft):
    coverage = keyword_recall(draft, case["expected_keywords"])
    normalized_draft = normalize_text(draft)
    forbidden_hits = [
        phrase for phrase in case["forbidden_keywords"]
        if normalize_text(phrase) in normalized_draft
    ]
    approved = coverage >= 0.75 and not forbidden_hits
    note = "Approved." if approved else "Missing coverage or contradiction detected."
    return {
        "status": "approved" if approved else "needs_revision",
        "coverage": coverage,
        "forbidden_hits": forbidden_hits,
        "note": note,
    }

def run_workflow(case, variant):
    run = {
        "case_id": case["id"],
        "variant": variant,
        "state": {
            "case_id": case["id"],
            "user_request": case["user_request"],
            "provided_context": deep_copy(case["provided_context"]),
        },
        "steps": [],
        "latency_s": 0.0,
        "cost_units": 0.0,
    }

    planned_roles = list(case["required_roles"])
    if variant == "over_delegating" and "researcher" not in planned_roles:
        planned_roles.insert(1, "researcher")

    planner_note = planner_handoff(
        case,
        variant,
        "researcher" if "researcher" in planned_roles else "writer",
        ["provided_context"] if case["provided_context"] else [],
    )
    append_step(
        run,
        "planner",
        reads={
            "user_request": case["user_request"],
            "provided_context": case["provided_context"],
        },
        writes={
            "plan": {
                "planned_roles": planned_roles,
                "success_criteria": case["must_include"],
            }
        },
        handoff=planner_note,
        summary="Planner chooses the route.",
    )

    if "researcher" in planned_roles:
        if case["research_topic"]:
            source = KNOWLEDGE_BASE[case["research_topic"]]
            research_brief = {
                "topic": case["research_topic"],
                "facts": deep_copy(source["facts"]),
                "source_docs": deep_copy(source["source_docs"]),
            }
        else:
            research_brief = {
                "topic": "provided_context",
                "facts": deep_copy(case["provided_context"]),
                "source_docs": ["provided_context"],
            }

        append_step(
            run,
            "researcher",
            reads={
                "plan": run["state"]["plan"],
                "provided_context": run["state"]["provided_context"],
            },
            writes={"research_brief": research_brief},
            handoff=researcher_handoff(case, variant, ["research_brief"]),
            summary="Researcher assembles a fact brief.",
        )

    source_key = "research_brief" if "research_brief" in run["state"] else "provided_context"
    actual_source = deep_copy(run["state"][source_key])
    used_source = deep_copy(actual_source)

    if source_key == "research_brief":
        facts = deep_copy(used_source["facts"])
    else:
        facts = deep_copy(used_source)

    if variant == "vague_handoff":
        facts = facts[:-1] if len(facts) > 1 else facts
        if source_key == "research_brief":
            used_source["facts"] = facts
        else:
            used_source = facts
    elif variant == "state_desync":
        corrupted_facts = deep_copy(facts)
        if corrupted_facts:
            corrupted_facts[0] = case["stale_fact"]
        if source_key == "research_brief":
            used_source["facts"] = corrupted_facts
        else:
            used_source = corrupted_facts
        facts = corrupted_facts

    draft = build_draft(case, facts)
    append_step(
        run,
        "writer",
        reads={
            "plan": run["state"]["plan"],
            source_key: used_source,
        },
        writes={
            "draft": draft,
            "draft_source_key": source_key,
        },
        handoff=writer_handoff(case, variant),
        summary="Writer produces a draft.",
    )

    initial_review = review_draft(case, run["state"]["draft"])

    if variant == "reviewer_loop":
        append_step(
            run,
            "reviewer",
            reads={"draft": run["state"]["draft"]},
            writes={
                "review_notes": {
                    "status": "needs_revision",
                    "coverage": initial_review["coverage"],
                    "forbidden_hits": [],
                    "note": "Please rewrite for style even though the facts already pass.",
                }
            },
            handoff=reviewer_revision_handoff(case),
            summary="Reviewer requests a redundant revision.",
        )

        append_step(
            run,
            "writer",
            reads={
                source_key: deep_copy(run["state"].get(source_key, [])),
                "review_notes": deep_copy(run["state"]["review_notes"]),
            },
            writes={"draft": run["state"]["draft"]},
            handoff=writer_handoff(case, "disciplined"),
            summary="Writer rewrites without changing the draft.",
        )

    final_review = review_draft(case, run["state"]["draft"])
    append_step(
        run,
        "reviewer",
        reads={"draft": run["state"]["draft"]},
        writes={
            "review_notes": final_review,
            "review_status": final_review["status"],
            "final_answer": run["state"]["draft"],
        },
        summary="Reviewer makes the final decision.",
    )

    run["final_state"] = deep_copy(run["state"])
    run["total_steps"] = len(run["steps"])
    return run

## Step 5: Generate a run corpus

This gives us a small but useful evaluation set: four tasks multiplied by five workflow policies.

In [ ]:
variants = [
    "disciplined",
    "vague_handoff",
    "over_delegating",
    "state_desync",
    "reviewer_loop",
]

workflow_runs = []
for case in workflow_cases:
    for variant in variants:
        workflow_runs.append(run_workflow(case, variant))

print("Workflow runs:", len(workflow_runs))
print("Variants:", ", ".join(variants))

## Step 6: Inspect one workflow trace

Readable traces are important. You should be able to audit a weak run before trusting any metric.

In [ ]:
def pretty_run(run):
    lines = []
    for index, step in enumerate(run["steps"], start=1):
        line = f"{index}. {step['role'].upper()} — {step['summary']}"
        if step["handoff"]:
            line += " | handoff to " + step["handoff"]["to"] + " => " + step["handoff"]["text"]
        if step["writes"]:
            line += " | writes: " + ", ".join(step["writes"].keys())
        lines.append(line)
    return "\n".join(lines)

example_run = next(
    run for run in workflow_runs
    if run["case_id"] == "incident_update" and run["variant"] == "reviewer_loop"
)
print(pretty_run(example_run))

## Step 7: Score handoff quality

For each handoff we ask:

1. Did the sender include the fields the receiver needs?
2. Did the message preserve critical task facts?
3. Did the sender point to the correct shared-state keys?

In [ ]:
def score_handoff(case, handoff, available_state):
    required_fields = HANDOFF_REQUIREMENTS.get(handoff["to"], [])
    present_fields = [
        field for field in required_fields
        if handoff["fields"].get(field) not in (None, "", [], {})
    ]
    completeness = round(len(present_fields) / len(required_fields), 3) if required_fields else 1.0

    note_blob = handoff["text"] + " " + json.dumps(handoff["fields"], sort_keys=True)
    context_score = keyword_recall(note_blob, case["must_include"][:3])

    if handoff["to"] == "researcher":
        state_reference = 1.0 if handoff["fields"].get("topic_or_context") else 0.0
    elif handoff["to"] == "writer":
        source_keys = handoff["fields"].get("source_keys", [])
        if isinstance(source_keys, str):
            source_keys = [source_keys]
        state_reference = 1.0 if source_keys and all(key in available_state for key in source_keys) else 0.0
    elif handoff["to"] == "reviewer":
        draft_key = handoff["fields"].get("draft_key")
        state_reference = 1.0 if draft_key and draft_key in available_state else 0.0
    else:
        state_reference = 0.0

    final_score = round(0.5 * completeness + 0.3 * context_score + 0.2 * state_reference, 3)
    return {
        "score": final_score,
        "completeness": completeness,
        "context_score": context_score,
        "state_reference": state_reference,
        "missing_fields": [field for field in required_fields if field not in present_fields],
    }

handoff_rows = []
for run in workflow_runs:
    case = next(item for item in workflow_cases if item["id"] == run["case_id"])
    for step_index, step in enumerate(run["steps"], start=1):
        if not step["handoff"]:
            continue
        scoring = score_handoff(case, step["handoff"], step["state_after"])
        handoff_rows.append(
            {
                "case_id": run["case_id"],
                "variant": run["variant"],
                "step_index": step_index,
                "from_role": step["handoff"]["from"],
                "to_role": step["handoff"]["to"],
                "handoff_score": scoring["score"],
                "missing_fields": ", ".join(scoring["missing_fields"]),
            }
        )

print(to_markdown_table(handoff_rows[:10], [
    "case_id",
    "variant",
    "step_index",
    "from_role",
    "to_role",
    "handoff_score",
    "missing_fields",
]))

## Step 8: Score delegation quality

Delegation is good when:

- the selected role is capable of the assigned objective
- the workflow uses only the roles it actually needs
- the ordering respects prerequisites such as research before writing

In [ ]:
def score_delegation(case, run):
    actual_roles = []
    for step in run["steps"]:
        if step["role"] not in actual_roles:
            actual_roles.append(step["role"])

    required_roles = case["required_roles"]
    extra_roles = [role for role in actual_roles if role not in required_roles]
    missing_roles = [role for role in required_roles if role not in actual_roles]

    capability_scores = []
    for step in run["steps"]:
        handoff = step.get("handoff")
        if not handoff:
            continue
        capability_scores.append(
            1.0 if objective_matches_role(handoff["to"], handoff["fields"].get("objective", "")) else 0.0
        )

    positions = {}
    for index, step in enumerate(run["steps"]):
        positions.setdefault(step["role"], index)

    order_checks = []
    if "researcher" in required_roles:
        order_checks.append(positions.get("researcher", 999) < positions.get("writer", 999))
    order_checks.append(positions.get("writer", 999) < positions.get("reviewer", 999))

    necessity_score = max(0.0, 1.0 - 0.25 * len(extra_roles) - 0.35 * len(missing_roles))
    capability_score = mean_or_zero(capability_scores)
    order_score = mean_or_zero(1.0 if passed else 0.0 for passed in order_checks)
    final_score = round(0.4 * necessity_score + 0.3 * capability_score + 0.3 * order_score, 3)

    return {
        "score": final_score,
        "extra_roles": extra_roles,
        "missing_roles": missing_roles,
        "necessity_score": round(necessity_score, 3),
        "capability_score": capability_score,
        "order_score": order_score,
    }

delegation_rows = []
for run in workflow_runs:
    case = next(item for item in workflow_cases if item["id"] == run["case_id"])
    scoring = score_delegation(case, run)
    delegation_rows.append(
        {
            "case_id": run["case_id"],
            "variant": run["variant"],
            "delegation_score": scoring["score"],
            "extra_roles": ", ".join(scoring["extra_roles"]),
            "missing_roles": ", ".join(scoring["missing_roles"]),
        }
    )

print(to_markdown_table(delegation_rows[:10], [
    "case_id",
    "variant",
    "delegation_score",
    "extra_roles",
    "missing_roles",
]))

## Step 9: Detect coordination failures

A good score is not enough. We also want explicit flags for common workflow bugs.

In [ ]:
def detect_coordination_failures(case, run):
    failures = []
    roles = [step["role"] for step in run["steps"]]

    if "researcher" in case["required_roles"] and roles.index("writer") < roles.index("researcher"):
        failures.append("writer_before_research")

    if "researcher" not in case["required_roles"] and "researcher" in roles:
        failures.append("unnecessary_research")

    if roles.count("reviewer") > 1:
        failures.append("reviewer_loop")

    for step in run["steps"]:
        if step["role"] == "writer" and "research_brief" in step["reads"]:
            if step["reads"]["research_brief"] != step["state_before"].get("research_brief"):
                failures.append("stale_shared_state")
                break

    if not run["final_state"].get("final_answer"):
        failures.append("missing_final_answer")

    if any(
        step.get("handoff")
        and step["handoff"]["to"] == "writer"
        and not step["handoff"]["fields"].get("source_keys")
        for step in run["steps"]
    ):
        failures.append("writer_handoff_without_sources")

    return failures

coordination_rows = []
for run in workflow_runs:
    case = next(item for item in workflow_cases if item["id"] == run["case_id"])
    failures = detect_coordination_failures(case, run)
    coordination_rows.append(
        {
            "case_id": run["case_id"],
            "variant": run["variant"],
            "failure_count": len(failures),
            "failures": ", ".join(failures),
        }
    )

print(to_markdown_table(coordination_rows[:10], [
    "case_id",
    "variant",
    "failure_count",
    "failures",
]))

## Step 10: Score shared-state correctness

Shared state is correct when:

- each agent reads the latest value from the shared store
- final state includes the expected artifacts
- the final answer is traceable to the latest draft

In [ ]:
def score_shared_state(case, run):
    read_checks = []
    mismatches = []

    for step in run["steps"]:
        for key, used_value in step["reads"].items():
            if key not in step["state_before"]:
                continue
            current_value = step["state_before"][key]
            matched = used_value == current_value
            read_checks.append(1.0 if matched else 0.0)
            if not matched:
                mismatches.append({"role": step["role"], "key": key})

    required_final_keys = ["plan", "draft", "review_notes", "final_answer"]
    if "researcher" in case["required_roles"] or "research_brief" in run["final_state"]:
        required_final_keys.append("research_brief")

    final_key_score = round(
        sum(1 for key in required_final_keys if key in run["final_state"]) / len(required_final_keys),
        3,
    )
    read_freshness = mean_or_zero(read_checks)
    lineage_score = 1.0 if run["final_state"].get("final_answer") == run["final_state"].get("draft") else 0.0
    final_score = round(0.55 * read_freshness + 0.25 * final_key_score + 0.2 * lineage_score, 3)

    return {
        "score": final_score,
        "read_freshness": read_freshness,
        "final_key_score": final_key_score,
        "lineage_score": lineage_score,
        "mismatches": mismatches,
    }

state_rows = []
for run in workflow_runs:
    case = next(item for item in workflow_cases if item["id"] == run["case_id"])
    scoring = score_shared_state(case, run)
    state_rows.append(
        {
            "case_id": run["case_id"],
            "variant": run["variant"],
            "shared_state_score": scoring["score"],
            "mismatch_count": len(scoring["mismatches"]),
        }
    )

print(to_markdown_table(state_rows[:10], [
    "case_id",
    "variant",
    "shared_state_score",
    "mismatch_count",
]))

## Step 11: Score end-to-end workflow success

The workflow still needs to produce a useful final answer. We use simple keyword coverage plus a review-status check.

In [ ]:
def score_workflow(case, run):
    final_answer = run["final_state"].get("final_answer", "")
    coverage = keyword_recall(final_answer, case["expected_keywords"])
    normalized_answer = normalize_text(final_answer)
    forbidden_hits = [
        phrase for phrase in case["forbidden_keywords"]
        if normalize_text(phrase) in normalized_answer
    ]
    review_score = 1.0 if run["final_state"].get("review_status") == "approved" else 0.25
    final_score = round((0.75 * coverage + 0.25 * review_score) * (0.5 if forbidden_hits else 1.0), 3)
    return {
        "score": final_score,
        "coverage": coverage,
        "review_score": review_score,
        "forbidden_hits": forbidden_hits,
    }

workflow_score_rows = []
for run in workflow_runs:
    case = next(item for item in workflow_cases if item["id"] == run["case_id"])
    scoring = score_workflow(case, run)
    workflow_score_rows.append(
        {
            "case_id": run["case_id"],
            "variant": run["variant"],
            "workflow_score": scoring["score"],
            "coverage": scoring["coverage"],
            "forbidden_hits": ", ".join(scoring["forbidden_hits"]),
        }
    )

print(to_markdown_table(workflow_score_rows[:10], [
    "case_id",
    "variant",
    "workflow_score",
    "coverage",
    "forbidden_hits",
]))

## Step 12: Combine the metrics into run summaries

Now we join handoff, delegation, coordination, shared-state, and task-success signals into one audit table.

In [ ]:
run_summaries = []

for run in workflow_runs:
    case = next(item for item in workflow_cases if item["id"] == run["case_id"])
    handoff_scores = [
        score_handoff(case, step["handoff"], step["state_after"])["score"]
        for step in run["steps"]
        if step.get("handoff")
    ]
    delegation = score_delegation(case, run)
    coordination_failures = detect_coordination_failures(case, run)
    shared_state = score_shared_state(case, run)
    workflow_quality = score_workflow(case, run)

    overall_quality = round(
        0.30 * workflow_quality["score"]
        + 0.25 * mean_or_zero(handoff_scores)
        + 0.20 * delegation["score"]
        + 0.25 * shared_state["score"]
        - 0.05 * len(coordination_failures),
        3,
    )

    run_summaries.append(
        {
            "case_id": run["case_id"],
            "variant": run["variant"],
            "handoff_score": mean_or_zero(handoff_scores),
            "delegation_score": delegation["score"],
            "shared_state_score": shared_state["score"],
            "workflow_score": workflow_quality["score"],
            "coordination_failures": len(coordination_failures),
            "overall_quality": overall_quality,
            "latency_s": round(run["latency_s"], 2),
            "cost_units": round(run["cost_units"], 3),
            "steps": run["total_steps"],
        }
    )

leaderboard = sorted(
    run_summaries,
    key=lambda row: (row["overall_quality"], -row["latency_s"]),
    reverse=True,
)

print(to_markdown_table(leaderboard[:12], [
    "case_id",
    "variant",
    "overall_quality",
    "handoff_score",
    "delegation_score",
    "shared_state_score",
    "workflow_score",
    "coordination_failures",
    "latency_s",
    "cost_units",
    "steps",
]))

## Step 13: Build trade-off tables by workflow policy

This is the key systems view: which policy gives the best quality, and what does it cost in latency and complexity?

In [ ]:
variant_summary = []
for variant in variants:
    rows = [row for row in run_summaries if row["variant"] == variant]
    variant_summary.append(
        {
            "variant": variant,
            "mean_quality": mean_or_zero(row["overall_quality"] for row in rows),
            "mean_handoff": mean_or_zero(row["handoff_score"] for row in rows),
            "mean_state": mean_or_zero(row["shared_state_score"] for row in rows),
            "mean_latency_s": round(statistics.fmean(row["latency_s"] for row in rows), 2),
            "mean_cost_units": round(statistics.fmean(row["cost_units"] for row in rows), 3),
            "mean_steps": round(statistics.fmean(row["steps"] for row in rows), 2),
            "mean_failures": round(statistics.fmean(row["coordination_failures"] for row in rows), 2),
        }
    )

variant_summary = sorted(variant_summary, key=lambda row: row["mean_quality"], reverse=True)
print(to_markdown_table(variant_summary, [
    "variant",
    "mean_quality",
    "mean_handoff",
    "mean_state",
    "mean_latency_s",
    "mean_cost_units",
    "mean_steps",
    "mean_failures",
]))

## Step 14: Compare explicit trade-offs

A workflow can be more expensive without being better. This table highlights the delta from the disciplined baseline.

In [ ]:
baseline_variant = next(row for row in variant_summary if row["variant"] == "disciplined")

tradeoff_rows = []
for row in variant_summary:
    tradeoff_rows.append(
        {
            "variant": row["variant"],
            "quality_delta_vs_disciplined": round(row["mean_quality"] - baseline_variant["mean_quality"], 3),
            "latency_delta_s": round(row["mean_latency_s"] - baseline_variant["mean_latency_s"], 2),
            "cost_delta": round(row["mean_cost_units"] - baseline_variant["mean_cost_units"], 3),
            "step_delta": round(row["mean_steps"] - baseline_variant["mean_steps"], 2),
        }
    )

print(to_markdown_table(tradeoff_rows, [
    "variant",
    "quality_delta_vs_disciplined",
    "latency_delta_s",
    "cost_delta",
    "step_delta",
]))

## Step 15: Bucket the failures

Failure buckets tell us what to fix next. A vague handoff and a stale-state bug need different interventions.

In [ ]:
failure_counter = Counter()
case_variant_failures = []

for run in workflow_runs:
    case = next(item for item in workflow_cases if item["id"] == run["case_id"])
    failures = detect_coordination_failures(case, run)
    workflow = score_workflow(case, run)
    if workflow["forbidden_hits"]:
        failures = failures + ["factual_contradiction"]
    if workflow["coverage"] < 1.0:
        failures = failures + ["missing_required_fact"]

    for failure in failures:
        failure_counter[failure] += 1

    case_variant_failures.append(
        {
            "case_id": run["case_id"],
            "variant": run["variant"],
            "failures": ", ".join(sorted(set(failures))),
        }
    )

bucket_rows = [
    {"failure_type": failure_type, "count": count}
    for failure_type, count in failure_counter.most_common()
]

print(to_markdown_table(bucket_rows, ["failure_type", "count"]))
print()
print(to_markdown_table(case_variant_failures[:10], ["case_id", "variant", "failures"]))

## Step 16: Optional — ask the local open-source model to critique the weakest handoff

Rule-based metrics stay primary. A local judge can still provide useful commentary on *why* a handoff is weak.

In [ ]:
weakest_handoff = min(handoff_rows, key=lambda row: row["handoff_score"])
weak_run = next(
    run for run in workflow_runs
    if run["case_id"] == weakest_handoff["case_id"] and run["variant"] == weakest_handoff["variant"]
)
weak_step = weak_run["steps"][weakest_handoff["step_index"] - 1]

judge_prompt = """
You are auditing a multi-agent handoff.
Give a short critique with:
1. what information is missing
2. what risk the missing information creates
3. one concrete improvement

Handoff sender: {sender}
Handoff receiver: {receiver}
Handoff text: {handoff_text}
Required fields for the receiver: {required_fields}
"""

print(generate(
    judge_prompt.format(
        sender=weak_step["handoff"]["from"],
        receiver=weak_step["handoff"]["to"],
        handoff_text=weak_step["handoff"]["text"],
        required_fields=HANDOFF_REQUIREMENTS[weak_step["handoff"]["to"]],
    ),
    max_new_tokens=160,
    temperature=0.1,
    do_sample=False,
))

## Step 17: Export artifacts

We save three useful outputs:

- run summaries for dashboards
- handoff-level audits for debugging
- a per-run failure report for error analysis

In [ ]:
summary_path = ARTIFACT_DIR / "multi_agent_run_summaries.json"
handoff_path = ARTIFACT_DIR / "multi_agent_handoffs.csv"
failure_path = ARTIFACT_DIR / "multi_agent_failures.json"

summary_path.write_text(json.dumps(run_summaries, indent=2), encoding="utf-8")

with open(handoff_path, "w", newline="", encoding="utf-8") as handle:
    writer = csv.DictWriter(handle, fieldnames=[
        "case_id",
        "variant",
        "step_index",
        "from_role",
        "to_role",
        "handoff_score",
        "missing_fields",
    ])
    writer.writeheader()
    writer.writerows(handoff_rows)

failure_records = []
for run in workflow_runs:
    case = next(item for item in workflow_cases if item["id"] == run["case_id"])
    failure_records.append(
        {
            "case_id": run["case_id"],
            "variant": run["variant"],
            "coordination_failures": detect_coordination_failures(case, run),
            "shared_state_mismatches": score_shared_state(case, run)["mismatches"],
            "workflow_forbidden_hits": score_workflow(case, run)["forbidden_hits"],
        }
    )

failure_path.write_text(json.dumps(failure_records, indent=2), encoding="utf-8")

print("Wrote", summary_path)
print("Wrote", handoff_path)
print("Wrote", failure_path)

## Recap

You now have a small, transparent evaluation harness for multi-agent workflows.

- **handoff quality** checks whether agents transfer the right task context
- **delegation quality** checks whether the right specialist was chosen
- **coordination failures** expose loops, unnecessary routing, and weak source passing
- **shared-state correctness** checks whether agents read the latest agreed state
- **trade-off tables** show when extra workflow complexity hurts more than it helps

This is the core measurement pattern you should reuse before scaling any planner-specialist-reviewer system.